Ma nem kell az Ollama. Futtasd sorban: **Shift + Enter**. Ha valami elakad: **Restart**, aztán **Run All**. A Copilot javaslata mellett **Keep** / **Undo**: a Keep csak azután, hogy elolvastad.

## 0. Előkészület

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd

from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Fent van a munkád a GitHubon?

A `.gitignore` tartalma. Ha még nincs meg: VS Code, bal oldalt **Explorer** → **New File…** → `.gitignore` (ponttal az elején, kiterjesztés nélkül, a `notebookok` mappa gyökerébe), ezt másold bele, és mentsd:

```text
# Ezek nem kerülnek fel a GitHubra
.env
.venv/
__pycache__/
.ipynb_checkpoints/
data/cache/
data/mnb/*.pdf
.DS_Store
```

Ha a repó elkészült és felkerült a GitHubra, futtasd ezt a cellát:

In [ ]:
def git(*parancs):
    """Lefuttat egy git-parancsot ebben a mappában. A kimenetét adja vissza, ha sikerült, különben None-t."""
    try:
        futas = subprocess.run(["git", *parancs], capture_output=True, text=True)
    except FileNotFoundError:
        return None
    return futas.stdout.strip() if futas.returncode == 0 else None


nem_kerulhet_fel = [".env", ".venv/pyvenv.cfg", "__pycache__/kurzus.cpython-314.pyc",
                    ".ipynb_checkpoints/minta-checkpoint.ipynb", "data/cache/minta.npy"]
fel_kell_kerulnie = "data/hicp_osszesen.csv"

gyoker = git("rev-parse", "--show-toplevel")
if gyoker is None:
    print("MÉG NINCS KÉSZ  ez a mappa még nem Git-repó (Source Control → Initialize Repository)")
elif Path(gyoker) == Path.home():
    print("HIBA     a repó a teljes felhasználói mappádat követi, nem a notebookok mappát. Szólj nekem.")
else:
    for fajl in nem_kerulhet_fel:
        kizarva = git("check-ignore", fajl) is not None
        print(f"RENDBEN  {fajl}: nem kerül fel" if kizarva else f"HIBA     {fajl}: felkerülne, hiányzik a .gitignore-ból")
    if git("check-ignore", fel_kell_kerulnie) is None:
        print(f"RENDBEN  {fel_kell_kerulnie}: felkerül")
    else:
        print(f"HIBA     {fel_kell_kerulnie}: a .gitignore az adatot is kizárja")
    print("RENDBEN  van commit" if git("rev-list", "--count", "HEAD") else "MÉG NINCS KÉSZ  még nincs commit")
    tavoli = git("remote", "get-url", "origin") or ""
    print("RENDBEN  fent van a GitHubon" if "github.com" in tavoli else "MÉG NINCS KÉSZ  még nincs fent a GitHubon (Publish Branch)")

### Feladat 1.1 – a te szavaiddal

Egy mondatban: mi a különbség a **commit** és a **push** között? Idézőjelek közé. Utána nyisd meg a repód oldalát a github.com-on, és mutasd meg nekem: ott a lakat (*Private*), és benne vannak a notebookok?

In [ ]:
commit_es_push = None  # a None helyére írd a mondatod, idézőjelek közé

In [ ]:
ellenoriz_feltetel("commit_es_push",
                   lambda s: isinstance(s, str) and len(s.split()) >= 8 and "commit" in s.lower() and "push" in s.lower(),
                   "egy egész mondat (legalább 8 szó), benne a commit és a push")

## 2. Táblázat beolvasása

A magyar havi infláció (HICP, éves változás, %), Eurostat. Az első 12 sor:

In [ ]:
adat = pd.read_csv("data/hicp_osszesen.csv", sep=";")
adat.head(12)

### Feladat 2.1 – olvasás

A táblázat 2023 januárjától 2025 novemberéig tart, havonta egy sorral. **Hány sora van?** Írd be a tipped a `None` helyére, aztán futtasd mindkét cellát.

In [ ]:
tipp_2_1 = None  # a None helyére írd a tipped

In [ ]:
# Olvasd el, és csak a tipped beírása után futtasd!
sorok_szama = len(adat)   # a len a sorok számát adja, mint a listánál az elemekét
print(sorok_szama)

ellenoriz("tipp_2_1", vart=sorok_szama)

Ha 35-öt tippeltél: egy hónap hiányzik. A 6. részben visszatérünk rá.

## 3. Egy oszlop, egy szám

In [ ]:
inflacio = adat["hicp_ev_ev"]   # egy oszlop, mint egy szótárelem

print(inflacio.max())     # a legnagyobb érték
print(inflacio.min())     # a legkisebb
print(inflacio.mean())    # az átlag
print(inflacio.iloc[0])   # az első sor értéke (sorszám szerint, 0-tól)

A táblázat vége (a `.tail(3)` az utolsó három sort mutatja):

In [ ]:
adat.tail(3)

### Feladat 3.1 – olvasás

Mennyi lesz az `adat["hicp_ev_ev"].iloc[-1]`? Melyik hónap értéke ez? (Az első órán a listánál a `-1` hátulról számolt.)

In [ ]:
tipp_3_1 = None  # a None helyére írd a tipped

In [ ]:
# Olvasd el, és csak a tipped beírása után futtasd!
utolso = adat["hicp_ev_ev"].iloc[-1]
print(adat["honap"].iloc[-1], utolso)

ellenoriz("tipp_3_1", vart=utolso)

## 4. Sorok kiválasztása feltétellel

In [ ]:
magas = adat["hicp_ev_ev"] > 10    # minden sorra: True vagy False
print(magas.head(12).tolist())

adat[magas]                         # csak azok a sorok, ahol True

Szövegre is lehet szűrni:

In [ ]:
adat[adat["honap"].str.startswith("2024")]

### Feladat 4.1 – olvasás

Hány sor marad, ha az infláció **20 fölötti** hónapjait választod ki? A 2. rész első 12 sorából ki tudod olvasni.

In [ ]:
tipp_4_1 = None  # a None helyére írd a tipped

In [ ]:
# Olvasd el, és csak a tipped beírása után futtasd!
nagyon_magas = adat[adat["hicp_ev_ev"] > 20]
print(nagyon_magas)

ellenoriz("tipp_4_1", vart=len(nagyon_magas))

### Feladat 4.2 – Copilottal

Állj a lenti üres cellába, nyomd meg a **Ctrl + I** (Mac: **⌘ + I**) billentyűket, és kérd ezt:

> Számold ki az `adat` táblából a 2024-es havi `hicp_ev_ev` értékek átlagát. Az eredmény az `atlag_2024` változóba kerüljön.

**Olvasd el, mielőtt elfogadod (Keep):** mire szűr? Hány sor marad a szűrés után?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz("atlag_2024", vart=adat[adat["honap"].str[:4] == "2024"]["hicp_ev_ev"].mean())

## 5. A Copilot ír, te elmagyarázod, te módosítod

### Feladat 5.1 – Copilottal

A lenti üres cellában **Ctrl + I** (Mac: **⌘ + I**):

> Számold ki az `adat` táblából évenként a havi `hicp_ev_ev` értékek átlagát. Az évet a `honap` oszlop első négy karakteréből vedd. Az eredmény az `evi_atlag` változóba kerüljön, és írd ki.

**Olvasd el, mielőtt elfogadod (Keep):** mi alapján csoportosít, és mit átlagol?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
def evenkent(x):
    """Évenként egy szám szótárban; mindegy, hogy a Copilot Series-t vagy kétoszlopos táblát adott."""
    if isinstance(x, pd.DataFrame):
        x = x.set_index(x.columns[0])[x.columns[-1]] if x.shape[1] > 1 else x.iloc[:, 0]
    return {str(ev): float(ertek) for ev, ertek in pd.Series(x).items()}


def egyezik(kapott, vart):
    """Ugyanazok az évek, és évenként legfeljebb 0,01 az eltérés."""
    try:
        k, v = evenkent(kapott), evenkent(vart)
    except (TypeError, ValueError):
        return False
    return k.keys() == v.keys() and all(abs(k[ev] - v[ev]) <= 0.01 for ev in v)


helyes_atlag = adat.groupby(adat["honap"].str[:4])["hicp_ev_ev"].mean()
ellenoriz_feltetel("evi_atlag", lambda x: egyezik(x, helyes_atlag),
                   "évenként a havi értékek átlaga (2023, 2024, 2025)")

### Feladat 5.2 – a te szavaiddal

Két-három mondatban: honnan veszi az évet, mire bontja a táblázatot, mit számol ki? Idézőjelek közé. (Ha párban dolgoztok, olvassátok fel egymásnak.)

In [ ]:
sajat_magyarazat = None  # a None helyére írd a magyarázatod, idézőjelek közé

In [ ]:
ellenoriz_feltetel("sajat_magyarazat",
                   lambda s: isinstance(s, str) and len(s.split()) >= 10 and "átlag" in s.lower() and "év" in s.lower(),
                   "legalább 10 szó, és szerepel benne az év és az átlag")

### Feladat 5.3 – egy sor, kézzel

Másold be a Copilot kódját a lenti cellába, és **kézzel, Copilot nélkül** írd át úgy, hogy évenként a **legmagasabb** havi értéket adja, az `evi_max` változóba. Ha a Copilot szürkén javasol, ne fogadd el: **Esc**.

In [ ]:
# Ide másold a Copilot kódját, és írj át benne egy sort

In [ ]:
ellenoriz_feltetel("evi_max", lambda x: egyezik(x, adat.groupby(adat["honap"].str[:4])["hicp_ev_ev"].max()),
                   "évenként a legmagasabb havi érték")

## 6. Ellenőrizd az asszisztenst: nőtt vagy csökkent?

Egy AI-asszisztens kódja erre a kérésre: „Számold ki, hány százalékponttal változott az infláció egy év alatt, 2025 novemberére.”

In [ ]:
# Az asszisztens kódja:
valtozas = adat["hicp_ev_ev"] - adat["hicp_ev_ev"].shift(12)   # az érték mínusz a 12 sorral korábbi érték
valtozas_2025_11 = valtozas.iloc[-1]                            # az utolsó sor: 2025 november
print(round(valtozas_2025_11, 1), "százalékpont")

Melyik hónappal hasonlította össze? A `shift(12)` 12 **sort** lép vissza, nem 12 hónapot. Nézd meg a 2025-06-os sort:

In [ ]:
adat.assign(osszehasonlitva_ezzel=adat["honap"].shift(12), valtozas=valtozas).tail(7)

### Feladat 6.1 – hibakeresés

A lenti üres cellában **Ctrl + I** (Mac: **⌘ + I**), és kérd ezt:

> Írj egy `egy_eves_valtozas(honap)` függvényt. Az `adat` táblából a `honap` oszlop alapján keresse ki a megadott hónap `hicp_ev_ev` értékét (pl. "2025-11") és az egy évvel korábbi hónapét (pl. "2024-11"), és a kettő különbségét adja vissza százalékpontban. Ne sorszám alapján keress, mert az adatból hiányozhat hónap.

Olvasd el, mielőtt elfogadod (Keep): hogyan állítja elő az egy évvel korábbi hónap nevét?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
# A helyes érték a hónapok neve alapján, a nyers adatból:
ertek_honapra = dict(zip(adat["honap"], adat["hicp_ev_ev"]))

ellenoriz_fuggveny("egy_eves_valtozas", [
    (("2025-11",), ertek_honapra["2025-11"] - ertek_honapra["2024-11"]),
    (("2025-06",), ertek_honapra["2025-06"] - ertek_honapra["2024-06"]),
    (("2024-03",), ertek_honapra["2024-03"] - ertek_honapra["2023-03"]),
])

**Ami lefut, még nem biztos, hogy jó. A hónap nevét nézd, ne a sorszámát.**

## 7. Otthonra: emlékszik az `AGENTS.md`?

Nézd meg, benne van-e az `AGENTS.md`-ben: „Elemzés előtt olvasd el a `data/adatszotar.md`-t.” Ha nincs, írd be a *Munkamód* alá, és mentsd.

In [ ]:
kategoriak = pd.read_csv("data/inflacio_kategoriak_2024.csv", sep=";")
kategoriak.head()

### Feladat 7.1 – Copilottal, a chatben

Copilot chat (**Ctrl + Alt + I**, Mac: **⌃ + ⌘ + I**), **Agent** módban, fájl hozzáadása **nélkül**:

> Számold ki a `kategoriak` táblából a közlekedési infláció 2024-es átlagát. Az eredmény az `atlag_kozlekedes` változóba kerüljön. A kódot a 7.1-es feladat üres cellájába írd.

**Olvasd el, mielőtt Keep-et nyomsz:** hány néven keresi a közlekedést? A válasz alatt, a **References** között ott van az `AGENTS.md`? Ha a kód nem a lenti cellába került, tedd át ide.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
# Az adatszótár szerinti helyes számítás: minden névváltozat, minden hónap egyszer.
kozlekedes_helyes = kategoriak[kategoriak["kategoria"].isin({"Közlekedés", "Kozlekedes"})].drop_duplicates("honap")
ellenoriz("atlag_kozlekedes", vart=kozlekedes_helyes["inflacio_ev_ev"].mean())

Ha `HIBA`: csak a `Közlekedés` sorokat nézte? Ott volt az `AGENTS.md` a References között?

## A végén: commit

A mai munkádat mentsd el a Gitbe is. Bal oldalt **Source Control**, fent az üzenet mezőbe írd be: `1. alkalom: pandas`, aztán **Commit**, végül **Sync Changes** (ez a push). A GitHubon a repód oldalán látnod kell az új commitot.